# Families: losses, gradients, offsets

This chapter describes every family available in `mboostrs`: the loss, its
negative gradient, the offset formula, and the response map. Read it after
[](prerequisites.ipynb) (exponential family, GLM link functions) and before
[](functional_gradient_boosting.ipynb) (how the engine uses the gradient and offset).

A **family** supplies three things to the boosting engine: the loss $\rho(y,f)$
(which defines the risk), a pseudo-residual $u$ fitted at each step, and the
initial constant $\hat f^{[0]}$. Many families obtain this constant by
minimizing $\sum_i w_i\rho(y_i,c)$; the offset conventions of individual families identify
exceptions. Usually $u=-\partial\rho/\partial f$; the Binomial section
records its constant gradient rescaling. The predictor $f$ is on the **link
scale**; the `response` map converts it back to the natural scale.

```{note}
Families define different targets: Gaussian boosting estimates the conditional
mean, Huber estimates a robust centre, Quantile estimates a chosen quantile,
and Binomial models the log-odds of a binary event. The base learners and
engine stay the same; the family supplies the loss, pseudo-residual $u$, and
risk.
```



## Response format and initial offsets

Every family fixes a **response domain**: the set of $y$ values for which its
loss is defined. `Binomial` takes $\{0, 1\}$, $\{-1, +1\}$ or proportions in
$[0, 1]$; `Poisson` and `NBinomial` take non-negative integers; `GammaReg`
and the survival families take positive reals; `PropOdds` takes ordered levels
coded $1 \dots K$. A response outside the accepted domain raises an error
naming the family and its required values, for example
`family 'Binomial' rejected response: y must be in {-1, +1}, {0, 1}, or [0, 1] proportions`.
Validate the response coding before interpreting a fitted loss.

Every family validates the response before fitting and computes an initial offset
$\hat f^{[0]}$. This offset is the starting point for the boosting loop: the first gradient is evaluated
relative to that constant fit.

For families whose offset minimizes the constant-fit risk,
$\hat f^{[0]} = \argmin_c \sum_i w_i\,\rho(y_i, c)$. For
families with a nuisance parameter (scale, threshold, dispersion), the offset
and the nuisance are obtained jointly by a one-dimensional numerical
optimization that matches R's `optimize` call exactly — the Brent minimizer
uses the same bracket and tolerance (`.Machine$double.eps^0.25`), so offsets
agree to $\sim 10^{-12}$.

`mboostrs` internally re-encodes the Binomial response from $\{0,1\}$ to
$\{-1,+1\}$. `AdaExp`, `HingeLoss`, and `AUC` also supply family-specific
response encoders. See the implementation note in
[](../implementation/parity_and_bugs).



## Mean and robust regression

```{list-table}
:header-rows: 1

* - Family
  - Loss $\rho(y,f)$
  - Neg. gradient $u$
  - Link / response
  - Offset
* - `Gaussian`
  - $\tfrac12 (y-f)^2$
  - $y - f$
  - identity / $\hat f$
  - weighted mean of $y$
* - `Laplace`
  - $|y - f|$
  - $\operatorname{sign}(y-f)$
  - identity / $\hat f$
  - weighted median of $y$
* - `Huber`
  - quadratic for $|y-f|\le\delta$, linear beyond
  - $y-f$ clipped to $[-\delta,\delta]$
  - identity / $\hat f$
  - numerical constant-fit optimum; adaptive $\delta$ = median absolute residual
* - `Quantile(τ)`
  - check loss $\rho_\tau(y-f)$
  - $\tau - \mathbf{1}\{y<f\}$
  - identity / $\hat f$
  - weighted $\tau$-quantile of $y$
* - `Expectile(τ)`
  - asymmetric squared error
  - $2|\tau - \mathbf{1}\{y<f\}|\,(y-f)$
  - identity / $\hat f$
  - mean of observations with $w_i=1$; weighted mean if none
```



### Deriving the offset

For a constant predictor, minimizing the weighted risk gives
$\hat c = \argmin_c \sum_i w_i \rho(y_i, c)$. The following derivations
distinguish this minimizer from the initial offset used by a family.

**Gaussian:** $\rho(y, c) = \tfrac12 (y - c)^2$. The first-order condition
is $\sum_i w_i (y_i - c) = 0$, giving $\hat c = (\sum w_i y_i) / (\sum w_i)$
— the weighted mean of $y$.

**Laplace:** $\rho(y, c) = |y - c|$. The minimizer is any median of the
weighted sample. With the convention of averaging at even $n$, the offset is
the weighted median using R's type-7 quantile (linear interpolation at
$(k-1)/(n-1)$). This requires sorting $y$ and accumulating weights to find
the crossing point.

**Quantile($\tau$):** $\rho_\tau(y, c) = (y - c)(\tau - \mathbf{1}\{y < c\})$.
The check loss is minimised at the weighted $\tau$-quantile. The offset
the fit uses is the weighted `qoffset` quantile, and `qoffset` defaults
to 0.5 even when $\tau$ is not. The value is still R's type-7 quantile.

**Expectile($\tau$):** $\rho(y, c) = |\tau - \mathbf{1}\{y < c\}| (y - c)^2$.
The first-order condition is $\sum_i w_i \, |\tau - \mathbf{1}\{y_i < c\}| \,
(y_i - c) = 0$, so $\hat c$ is the weighted mean of $y$ with asymmetric
weights: observations below $c$ receive weight $w_i(1-\tau)$, and those
above receive $w_i\tau$. Because these weights depend on $c$, the expectile
can be found by solving this piecewise-linear first-order condition.
The offset used by the code starts at the mean of observations with $w_i=1$, independently of $\tau$, and uses the weighted mean if there are no such observations. This initial
offset generally differs from the constant-fit expectile; the subsequent
gradients use the asymmetric weights from the derivation.

**Huber:** $\rho(y,c)$ is quadratic for $|y-c|\le\delta$ and linear beyond.
The Huber offset uses a Brent minimizer of the weighted constant-fit risk over
$[\min_i y_i,\max_i y_i]$. The threshold for this optimization is the supplied
$\delta$, or, by default, the unweighted median of $|y_i|$ with a zero median
replaced by $10^{-10}$. This optimum generally differs from the weighted median.
With the adaptive default, `calibrate` subsequently sets the risk threshold
to the unweighted median of $|y_i-f_i|$, with the same zero-median fallback. The
recalibrated loss can therefore differ from the loss minimized for the offset.



### Choosing among the mean and robust losses

- **Laplace** targets the conditional median. It is insensitive to outliers in
  $y$ (the gradient is bounded by $\pm 1$). Use it when you want a robust
  centre estimate without the adaptive threshold machinery of Huber. The offset
  is the weighted median, computed by sorting the weighted sample — matching R's
  `quantile(type=7)` convention.

- **Huber** uses a quadratic loss for small residuals and a linear loss for
  large residuals. With the adaptive default, `calibrate` sets $\delta$ to the unweighted median
  absolute residual. This is a single threshold for the sample; it does not
  estimate a separate scale at each covariate value. The implementation uses
  median selection rather than sorting the full residual array. The gradient
  threshold can lag the risk threshold; see The `calibrate` mechanism.

- **Quantile** targets an arbitrary quantile ($\tau = 0.5$ gives the median;
  other values give lower/upper quantiles). The check loss is piecewise linear,
  so the gradient is piecewise constant — each base learner fits a sign-weighted
  least squares. Useful for prediction intervals and heteroscedasticity analysis.

- **Expectile** is the asymmetric squared-error analogue of quantiles. Every
  observation contributes (unlike quantile, where only the sign matters), so the
  fit is smoother. An expectile at level $\tau$ is the minimizer of an asymmetric squared error: each residual is weighted by $\tau$ when the observation lies above the fit and by $1-\tau$ when it lies below.

```{note}
The initial offset is computed before the boosting loop. Its convention
depends on the family; in particular, the expectile family starts at a mean
rather than solving for the constant-fit expectile.
```



## Binary and exponential-family classification

For a binary response $y\in\{0,1\}$, write $\eta$ for the full log-odds and
$p=(1+e^{-\eta})^{-1}$. The Bernoulli negative log-likelihood and its negative
gradient with respect to $\eta$ are

$$
\rho_{\mathrm{Bern}}(y,\eta)=-[y\log p+(1-y)\log(1-p)],
\qquad -\frac{\partial\rho_{\mathrm{Bern}}}{\partial\eta}=y-p.
$$

`Binomial` uses the half-log-odds predictor $f=\eta/2$, with response map
$p=\sigma(2f)=(1+e^{-2f})^{-1}$. The family re-encodes the response as
$y^*=2y-1\in\{-1,+1\}$. The encoding and the scale of the predictor are
separate choices: changing the response labels alone does not halve the logit.

```{note}
For binary observations, both $\{0,1\}$ and $\{-1,+1\}$ labels are accepted.
`Binomial::encode_response` maps $0$ to $-1$ and preserves $-1$ and $+1$, so
both codings produce the same internal response. The formulas here concern
binary labels; they do not derive the implementation's behavior for proportions.
```

### Half-log-odds parameterization

Substituting $\eta=2f$ in the Bernoulli loss gives

$$
\rho_{\mathrm{Bern}}(y^*,f)=\log(1+e^{-2y^*f}),
\qquad
-\frac{\partial\rho_{\mathrm{Bern}}}{\partial f}
=\frac{2y^*}{1+e^{2y^*f}}=2(y-p).
$$

In `crates/mboostrs-core/src/families.rs`, `Binomial::risk` reports the
natural-log risk $\sum_i w_i\rho_{\mathrm{Bern}}(y_i^*,f_i)$.
`Binomial::negative_gradient` and the fused `ngradient_and_risk` method use
an additional constant factor $1/\log 2$ in the pseudo-residual:

$$
u_{\mathrm{impl}}=
\frac{2y^*}{(\log 2)(1+e^{2y^*f})}
=\frac{2(y-p)}{\log 2}.
$$

Thus the pseudo-residual is the negative derivative of the Bernoulli loss
expressed in base-2 logarithms. It has the same descent direction as the
reported natural-log loss, with a constant rescaling of the update. This
factor affects the path for a fixed step length $\nu$; the gradient is not
$\tfrac12(y-p)$. The implementation evaluates the risk and gradient at
$f$ clipped to $[-36,36]$ to keep exponentials finite; the derivative identities
of this section describe the unclipped loss.

For the initial offset, let $\bar y_w=\sum_i w_i y_i/\sum_i w_i$. The
constant-fit first-order condition is

$$
\sum_i w_i(y_i-p(c))=0,
\qquad p(c)=\bar y_w,
\qquad c=\tfrac12\log\frac{\bar y_w}{1-\bar y_w}.
$$

This is half the usual log-odds. In the code, $\bar y_w$ is clipped away from
0 and 1 so that the offset remains finite for a sample containing only one
class. Every update is added on the $f$ scale, and prediction applies
$\sigma(2f)$. This parameterization determines the response map; it does not
imply that a finitely stopped boosting fit equals an unregularized logistic fit.

`AdaExp` uses the exponential loss $\rho=e^{-(2y-1)f}$ (AdaBoost), with
negative gradient $(2y-1)e^{-(2y-1)f}$. The exponential can grow rapidly for
strongly misclassified observations.


## Counts

```{list-table}
:header-rows: 1

* - Family
  - Mean model
  - Neg. gradient
  - Offset
* - `Poisson`
  - $\mu = e^{f}$, Poisson deviance
  - $y - e^{f}$
  - $\log(\bar y_w)$
* - `NBinomial`
  - $\mu = e^{f}$, dispersion $\sigma$ nuisance
  - $y - (y+\sigma)\,\mu/(\mu+\sigma)$
  - $\log(\hat\mu)$ (numerical, with $\sigma$ estimated jointly)
```

`Poisson` is the standard count family. The offset is the log of the weighted
mean rate; the negative gradient $y - \hat\mu$ is the raw residual on the
response scale.

`NBinomial` accommodates overdispersion via a scalar dispersion $\sigma$ that is
profiled out each iteration. The negative binomial probability is

$$
P(Y=y) = \binom{y+\sigma-1}{y}\big(\tfrac{\sigma}{\mu+\sigma}\big)^\sigma
\big(\tfrac{\mu}{\mu+\sigma}\big)^y,
$$

and $\sigma$ is estimated by maximizing the profile likelihood (a 1-D Brent
search, matching R's `optimize`). The negative gradient depends on $\sigma$,
which is re-estimated each iteration via `calibrate`.



## Positive and skewed responses

`GammaReg` models a positive response with a log link, $\mu = e^f$, using the
Gamma deviance

$$
\rho(y, f) = \frac{y}{\mu} - \log\frac{y}{\mu} - 1,
\qquad u = \frac{y - \mu}{\mu},
$$

with shape as a nuisance parameter estimated by `calibrate`. The offset is
$\hat f^{[0]} = \log(\bar y_w)$. The negative gradient is the deviance residual
on the link scale.

`GaussReg` (not available in R `mboost`) regresses on $y$ with an identity link
but estimates a residual variance nuisance parameter $\sigma^2$ — it is the
Gaussian family with a separate scale estimate, used in distributional (GAMLSS)
models where both $\mu$ and $\sigma$ are modelled.



## Ordinal

`PropOdds` fits a proportional-odds cumulative model: a single predictor $f$
shifts a set of ordered thresholds $\theta_1 < \dots < \theta_{K-1}$, giving
cumulative probabilities $P(y \le k) = (1+e^{-(\theta_k - f)})^{-1}$. The
thresholds are nuisance parameters estimated alongside the boosting steps. The
response must be an integer-coded ordered factor or 0-based integer classes.

**Response format:** integer-valued $y \in \{0, 1, \dots, K-1\}$, where the
categories are ordered. Internally, the family builds $K-1$ threshold parameters
estimated during `calibrate`. `predict(type="response")` returns an
$(n, K)$ probability matrix with columns for each class.



## Survival

`CoxPH`, `Weibull`, and `Loglog` take a right-censored `Surv(time, status)`
response and are treated in their own chapter — see [](survival.ipynb) for the
likelihood, gradient, and censoring format.



## Multi-class and distributional

`Multinomial` (see [](multinomial.ipynb)) generalizes Binomial to $K$ classes with
a Kronecker-structured design. The `*LSS` families (see [](gamlss.ipynb)) model
**every** parameter of a distribution and have their own multi-predictor engine.



## The inverse link table

The following table gives the full inverse link (`response`) for every family,
since the negative gradient is expressed on the link scale but prediction
requires mapping back to the natural scale:

| Family | Link $g(\mu)$ | Inverse link $\mu = g^{-1}(f)$ | `predict(type="response")` |
|---|---|---|---|
| Gaussian, GaussReg, Laplace, Huber, Quantile, Expectile | identity | $\mu = f$ | $f$ |
| Binomial | half-logit | $\mu = (1+e^{-2f})^{-1}$ | $P(y=1)$ |
| Poisson | log | $\mu = e^{f}$ | expected count |
| NBinomial | log | $\mu = e^{f}$ | expected count |
| GammaReg | log | $\mu = e^{f}$ | expected value |
| AdaExp | exponential | margin $= 2y-1$ | sign(P(class=+1)) |
| PropOdds | logit | cumulative $P(y \le k)$ | class probabilities |
| CoxPH | log hazard ratio | $\exp(f)$ | (relative risk) |
| Weibull, Loglog | identity / log | see [](survival.ipynb) | location / scale |
| Multinomial | logit | softmax | class probabilities |
| *LSS families* | per-parameter | per-parameter | per-parameter |

Families with a scale or threshold nuisance (Huber, NBinomial, PropOdds,
Weibull, Loglog) estimate these via the `calibrate` method on each iteration;
the nuisance is stored in the boosting path and used for risk evaluation and
prediction.



## Choosing a family: practical guidance

Choosing the family determines what aspect of the conditional distribution
the model targets. The
base learners and the algorithm are the same regardless; only the loss and
gradient change.

**Continuous response:**
- Start with `Gaussian`. If the residual plot shows heavy tails or outliers,
  consider `Huber` (robust mean) or `Quantile(0.5)` (median).
- If you need prediction intervals, fit separate `Quantile(0.1)` and
  `Quantile(0.9)` models (no distributional assumption required).
- If you want to model dispersion, use `GaussianLSS` (see [](gamlss.ipynb)).

**Binary response:**
- Use `Binomial`. `AdaExp` is strictly less numerically stable and included only
  for parity with R.

**Count response:**
- Start with `Poisson`. If the variance exceeds the mean (overdispersion), switch
  to `NBinomial`.
- To model both rate and dispersion, use `NBinomialLSS`.

**Positive, right-skewed response:**
- Use `GammaReg` (mean model) or `GammaLSS` (mean and shape).

**Ordered categories:**
- Use `PropOdds` for ordered y with $K > 2$ classes.

**Time-to-event (censored):**
- Use `CoxPH` for proportional hazards or `Weibull`/`Loglog` for parametric AFT
  models. For covariate-varying scale, use the `*LSS` versions.

```{note}
Several `mboostrs` families deliberately differ from R `mboost`/`gamboostLSS`
(LogLog's $\sigma$ offset, Weibull's $\mu$ offset, and ZINB's gradient).
These are documented in [](../implementation/parity_and_bugs).
```



## The `calibrate` mechanism

Families with a **nuisance parameter** (scale $\sigma$, threshold $\delta$,
class thresholds $\theta_k$) estimate it via `calibrate`, a method called
**every boosting iteration** after the update. It takes the current predictor
$f^{[m]}$, the response $y$, and weights $w$, and returns an updated nuisance
parameter value for use in risk evaluation and subsequent gradient computations.

For likelihood families, a nuisance parameter can be estimated by
**profile likelihood**: at the current predictor $f^{[m]}$, maximize the
log-likelihood over the nuisance $\phi$ only. Huber instead uses a residual
summary. The profile-likelihood calculation is:

$$
\hat\phi^{[m]} = \argmax_\phi \sum_i w_i \, \ell(y_i, f_i^{[m]}, \phi).
$$

The families that use `calibrate` are:

| Family | Nuisance | How it is estimated | Frequency |
|---|---|---|---|
| Huber | $\delta$ (threshold) | unweighted median of $|y-f|$; $10^{-10}$ if the median is zero | every iteration for adaptive $\delta$ |
| NBinomial | $\sigma$ (dispersion) | 1-D Brent maximizer of profile NLL | every iteration |
| GammaReg | $\alpha$ (shape) | Brent maximizer of profile Gamma log-likelihood | every iteration |
| PropOdds | $\theta_1,\dots,\theta_{K-1}$ (thresholds) | Brent on each threshold conditional on $f$ | every iteration |
| CoxPH | baseline hazard $\Lambda_0(t)$ | Breslow estimator {cite}`breslow1972contribution` (cumulative sum over risk sets) | every iteration |
| Weibull, Loglog | $\sigma$ (scale) | Brent on profile likelihood | every iteration |

The scalar profile-likelihood searches use a one-dimensional Brent
optimizer; Huber's median calculation is not a likelihood optimization.

For Huber, `calibrate` sets the threshold used by `risk`. The threshold for the gradient is computed from the predictor passed to the previous gradient call, with the zero predictor used at the first call. With the adaptive default, the gradient threshold can therefore lag the risk threshold. With a fixed $\delta$, the supplied value controls the offset and the risk, while the gradient threshold still follows the previous-predictor rule. The clipped-residual derivative below describes a Huber loss at a fixed threshold.



### Per-iteration calibration

The nuisance parameter interacts with the risk and gradient. For NBinomial:

$$
\rho(y, f, \sigma) = -\log\binom{y+\sigma-1}{y} - \sigma\log\frac{\sigma}{\mu+\sigma}
                     - y\log\frac{\mu}{\mu+\sigma}, \qquad \mu = e^f.
$$

The gradient $u = y - (y+\sigma)\mu/(\mu+\sigma)$ depends on $\sigma$, and
the risk $\sum w_i \rho(y_i, f_i, \sigma)$ depends on $\sigma$ through the
log-likelihood. If $\sigma$ were fixed at its offset value, the gradient
would become progressively mis-calibrated as $f$ moves away from the offset.
Re-estimating $\sigma$ each iteration keeps the gradient aligned with the
current fit — it is the maximum likelihood estimate of dispersion conditional
on the current linear predictor.



### The `ngradient_and_risk` fusion

The engine requests the negative gradient and risk through a single interface:

$$
\texttt{ngradient\_and\_risk}(y, f, w) \to (u, R),
$$

where $R=\sum_i w_i\rho(y_i,f_i)$. After a predictor update, the engine
calls `calibrate` before evaluating the gradient and risk at the updated fit.

Families that override this method can share link evaluations and other
intermediate quantities between the two calculations. The trait's default
method calls the gradient and risk separately, so this interface does not
imply a fixed reduction in total fitting time. An illustrative Gaussian
implementation is:

```rust
fn ngradient_and_risk(&self, y: &[f64], f: &[f64], w: &[f64]) -> (Vec<f64>, f64) {
    let mut u = vec![0.0; y.len()];
    let mut risk = 0.0;
    for i in 0..y.len() {
        let r = y[i] - f[i];          // example: Gaussian
        u[i] = r;
        risk += w[i] * r * r / 2.0;
    }
    (u, risk)
}
```


## Ranking and concordance losses

Three families optimize a **ranking** objective rather than a per-observation
loss: `AUC` (area under the ROC curve), `Gehan` (accelerated-failure-time rank
loss), and `Cindex` (Uno's concordance index). Their risk is a sum over
**pairs** of observations, so a single gradient evaluation is $O(n^2)$ rather
than $O(n)$. All three use a constant `offset` of $0$ (the loss depends only on
differences $f_i - f_j$, so a constant predictor is uninformative).

### `AUC` — $(1-\mathrm{AUC})$ loss

`AUC` maximizes the area under the ROC curve for a two-level factor response,
encoded internally to $y \in \{-1, +1\}$ ($\mathrm{pos} = \{i : y_i = +1\}$,
$\mathrm{neg} = \{j : y_j = -1\}$). The AUC counts correctly-ordered
positive/negative pairs, which is discontinuous, so the gradient uses a smoothed
surrogate. With standardized scores $f \gets f / \operatorname{sd}(f)$ and
$M_{ji} = f_i - f_j$ for $i \in \mathrm{pos}, j \in \mathrm{neg}$:

$$
\text{smoothed loss} = \sum_{i \in \mathrm{pos}} \sum_{j \in \mathrm{neg}}
  \ell_{\mathrm{approx}}(M_{ji}), \qquad
u_i = \begin{cases}
  \sum_{j \in \mathrm{neg}} \ell'_{\mathrm{approx}}(M_{ji}) & i \in \mathrm{pos},\\
 -\sum_{i \in \mathrm{pos}} \ell'_{\mathrm{approx}}(M_{ji}) & i \in \mathrm{neg},
\end{cases}
$$

where $\ell_{\mathrm{approx}}$ is a clipped-quadratic ramp that is $0$ for
$x < -1$, $1$ for $x > 1$, and smooth in between. The reported **risk** is the
true, unsmoothed $1 - \mathrm{AUC} = 1 - \tfrac{1}{|\mathrm{pos}||\mathrm{neg}|}
\sum_{i,j} \mathbf{1}\{f_i > f_j\}$, used for stopping and cross-validation.

### `Gehan` — AFT rank loss

`Gehan` fits an accelerated-failure-time model by minimizing a censored rank
statistic. For a `Surv(time, status)` response with $Z_i = \log t_i$,
$\delta_i = \text{status}_i$, and residuals $r_i = Z_i - f_i$:

$$
\rho = \frac{1}{n} \sum_i \delta_i \!\!\sum_{j : r_j \ge r_i}\!\! (r_j - r_i),
\qquad
u_i = -\frac{1}{n}\Big(\delta_i \, \#\{j : r_j \ge r_i\}
       - \#\{j : \delta_j = 1, \, r_j \le r_i\}\Big).
$$

The loss depends only on pairwise **residual differences**
$r_j - r_i = Z_j - Z_i - (f_j - f_i)$, in which a constant added to $f$ cancels
— so the intercept is **not identifiable** (the risk is flat in a constant
offset). `mboostrs` returns `offset = 0` by design; fitted values match R up to
an additive constant, and all identifiable quantities (risk, selected learners,
coefficient increments, centered fitted values) match exactly.

### `Cindex` — Uno's concordance ($\sigma$)

`Cindex(sigma=0.1)` maximizes a smoothed, inverse-probability-of-censoring-weighted
(IPCW) estimate of the concordance probability. For a `Surv` response it builds
an $n \times n$ weight matrix $W_{ij} = \hat G_i^{-2} \,
\mathbf{1}\{t_i < t_j\} \, w_i w_j$ (normalized to sum $1$), where
$\hat G_i$ is the reverse-Kaplan-Meier estimate of the censoring survival at
$t_i$ (censored observations get weight $0$). With $\eta_{ij} = f_i - f_j$ and a
logistic smoother of scale $\sigma$:

$$
\rho = -\sum_{i,j} W_{ij}\,\ell_\sigma(\eta_{ij}), \quad
\ell_\sigma(x) = \frac{1}{1 + e^{x/\sigma}}, \quad
u_k = \sum_i W_{ik}\ell'_\sigma(\eta_{ik}) - \sum_j W_{kj}\ell'_\sigma(\eta_{kj}).
$$

The smoothing scale $\sigma$ (default $0.1$) trades bias for gradient
smoothness. `Cindex` requires $\{0,1\}$ weights and so supports subsampling
cross-validation and stability selection.

```{note}
`AUC` and `Gehan` also accept **non-unit weights** through a principled weighted
gradient (each observation carries weight $w_i$; the gradient gains a $w_i$
factor and remains the exact negative gradient of the weighted risk). This
enables `cvrisk`/`stabsel` for both. R's weighted `ngradient` for these families does not return the exact negative gradient of the weighted risk. `mboostrs` computes that exact gradient for weighted fits, so its weighted results differ from R's; unit-weight fits match R bit for bit. See [](../implementation/parity_and_bugs).
```


## AFT survival families

`Weibull`, `Loglog` and `Lognormal` are accelerated failure time (AFT) models: the predictor $f$ shifts the log of the time scale. Their losses, negative gradients and the scale of the predictor for each family are given in [](survival.ipynb).



## Custom families

The built-in families cover the standard `mboost` / `gamboostLSS` catalog.
Each Python family class (`Gaussian`, `Binomial`, …) is a **thin wrapper** that
maps to a high-performance Rust implementation of the loss, negative gradient,
offset, and risk in `crates/mboostrs-core/src/families.rs`. The boosting engine calls these methods in its inner loop. The compiler generates a separate copy of that loop for each family type, so the calls are resolved at compile time; [](../implementation/architecture) describes the mechanism.

Adding a **new family** therefore requires implementing the loss, gradient, and
offset in Rust (in `families.rs` for single-parameter families, or `lss.rs` for
distributional families) and registering the Python wrapper class in
`python/mboostrs/families.py`. The `Family` trait interface is:

```rust
pub trait Family {
    fn offset(&self, y: &[f64], w: &[f64]) -> f64;
    fn negative_gradient(&self, y: &[f64], f: &[f64], w: &[f64], out: &mut [f64]);
    fn risk(&self, y: &[f64], f: &[f64], w: &[f64]) -> f64;
    fn response(&self, f: &[f64], out: &mut [f64]);  // inverse link
    fn check_y(&self, y: &[f64]) -> Result<(), MboostrsError>;
    // Fused gradient + risk (the engine's hot path):
    fn ngradient_and_risk(&self, y: &[f64], f: &[f64], w: &[f64]) -> (Vec<f64>, f64);
    // Optional per-iteration nuisance update:
    fn calibrate(&mut self, y: &[f64], f: &[f64], w: &[f64]) { }
}
```

## Exercises

````{admonition} Exercise: Binomial coding and predictor scale
:class: dropdown

For $y\in\{0,1\}$, let $\eta$ be the full logit,
$p=(1+e^{-\eta})^{-1}$, and let

$$
\rho(y,\eta)=-[y\log p+(1-y)\log(1-p)].
$$

**a)** Derive $-\partial\rho/\partial\eta=y-p$.

**b)** Minimize $\sum_i w_i\rho(y_i,c)$ and obtain the full-logit offset
$c=\log(\bar y_w/(1-\bar y_w))$.

**c)** Set $y^*=2y-1$ and $f=\eta/2$. Derive the loss, offset, and negative
gradient on the $f$ scale, and compare with the implementation's extra
$1/\log 2$ gradient factor.
````

````{admonition} Solution
:class: dropdown

**a)** Since $\partial p/\partial\eta=p(1-p)$,

$$
\frac{\partial\rho}{\partial\eta}
=-\left[\frac{y}{p}-\frac{1-y}{1-p}\right]p(1-p)=p-y,
\qquad u_\eta=y-p.
$$

The negative gradient of a loss equals the positive derivative of the
corresponding log-likelihood.

**b)** The constant-fit derivative is
$\sum_i w_i[p(c)-y_i]$. Setting it to zero gives $p(c)=\bar y_w$, hence

$$
c=\log\frac{\bar y_w}{1-\bar y_w}.
$$

This formula requires $0<\bar y_w<1$; otherwise the unpenalized optimum is
at an infinite logit.

**c)** Replacing $\eta$ by $2f$ gives

$$
\rho(y^*,f)=\log(1+e^{-2y^*f}),
\qquad
-\frac{\partial\rho}{\partial f}
=\frac{2y^*}{1+e^{2y^*f}}=2(y-p).
$$

The offset is now $f^{[0]}=\tfrac12\log(\bar y_w/(1-\bar y_w))$ because
$f=\eta/2$, and the inverse link is $p=\sigma(2f)$. Response re-encoding
alone would leave the full-logit offset unchanged. The implementation's
pseudo-residual is $u_{\mathrm{impl}}=2(y-p)/\log 2$, the negative derivative
of $\rho/\log 2$, while its reported risk uses $\rho$.
````

````{admonition} Exercise: Poisson negative gradient and offset
:class: dropdown

The Poisson negative log-likelihood, up to a term depending only on $y$, is

$$
\rho(y,f)=\mu-y\log\mu,\qquad\mu=e^f.
$$

**a)** Derive the negative gradient $u=-\partial\rho/\partial f$.

**b)** Show that the constant-fit offset is $f^{[0]}=\log\bar y_w$ when
$\bar y_w>0$.

**c)** The negative-binomial negative gradient is
$u=y-(y+\sigma)\mu/(\mu+\sigma)$. Show that it approaches the Poisson
gradient as $\sigma\to\infty$.
````

````{admonition} Solution
:class: dropdown

**a)** Since $\partial\mu/\partial f=\mu$,

$$
\frac{\partial\rho}{\partial f}
=\mu-\frac{y}{\mu}\mu=\mu-y,
\qquad u=y-\mu.
$$

**b)** Differentiating the constant-fit risk gives

$$
\sum_i w_i(e^c-y_i)=0,
\qquad e^c=\bar y_w,
\qquad c=\log\bar y_w.
$$

If every positive-weight response is zero, the infimum occurs as
$c\to-\infty$; the current implementation returns the finite fallback $0$.

**c)** Rewrite the negative-binomial gradient term as

$$
\frac{(y+\sigma)\mu}{\mu+\sigma}
=\mu\frac{1+y/\sigma}{1+\mu/\sigma}\longrightarrow\mu.
$$

Hence $u_{\mathrm{NB}}\to y-\mu=u_{\mathrm{Pois}}$.
````

````{admonition} Exercise: The Huber family bridges L₁ and L₂
:class: dropdown

The Huber loss with threshold $\delta > 0$ is

$$
\rho_\delta(r) =
\begin{cases}
\frac12 r^2,                     & |r| \le \delta, \\[4pt]
\delta |r| - \frac12 \delta^2,   & |r| > \delta,
\end{cases}
\qquad r = y - f.
$$

**a)** Show that as $\delta \to \infty$, $\rho_\delta(r) \to \frac12 r^2$
(the Gaussian / L₂ loss) uniformly on compact sets.

**b)** Show that as $\delta \to 0$, $\rho_\delta(r) / \delta \to |r|$
(the Laplace / L₁ loss).

**c)** Derive the negative gradient and confirm it is the clipped residual:

$$
u_\delta(r) =
\begin{cases}
r,     & |r| \le \delta, \\[4pt]
\delta \operatorname{sign}(r), & |r| > \delta.
\end{cases}
$$

**d)** Explain how the adaptive threshold changes as residuals change.
What is its current calibration rule? What happens to a Huber loss with a
threshold that is large or small relative to the residuals?

````

````{admonition} Solution
:class: dropdown

**a)** For any finite $r$, choose $\delta > |r|$; then $\rho_\delta(r) = \frac12 r^2$.
As $\delta \to \infty$, this holds for all $r$, pointwise.  The convergence is
uniform on any compact set $[-M, M]$ because $\delta \ge M$ suffices for every
$r$ in the set.

**b)** For $r \ne 0$ and $\delta < |r|$,

$$
\frac{\rho_\delta(r)}{\delta} = \frac{\delta|r| - \frac12\delta^2}{\delta}
 = |r| - \frac12 \delta \to |r| \quad \text{as } \delta \to 0.
$$

For $r=0$, both sides are $0$.  So $\rho_\delta(r)/\delta$ converges to
$|r|$ pointwise.

**c)** Differentiating:

$$
u = -\frac{\partial\rho}{\partial f} = \frac{\partial\rho}{\partial r}
= \begin{cases}
r,                       & |r| \le \delta, \\[4pt]
\delta \operatorname{sign}(r), & |r| > \delta.
\end{cases}
$$

This is the residual $r$ capped at $\pm \delta$ — hence the gradient is bounded,
making Huber robust to large outliers.

**d)** The threshold $\delta$ determines which residuals receive quadratic
loss and which receive linear loss. The adaptive calibration rule uses
$\operatorname{median}_i|y_i-f_i|$, replacing a zero median by $10^{-10}$,
so the risk threshold tracks the residual magnitude as the fit changes. It is an unweighted median
absolute residual, without centering the residuals at their median or
multiplying by 1.4826. A fixed positive threshold also defines a valid Huber
loss; re-estimation is an adaptive convention rather than a mathematical
requirement. The gradient threshold can lag the risk threshold (see The `calibrate` mechanism).

If $\delta$ exceeds every absolute residual, the sample loss is quadratic.
If $\delta$ is small relative to most residuals, those residuals lie in the
linear tails and their pseudo-residuals have magnitude $\delta$ for a Huber
loss evaluated at that threshold.
````


## Numerical check: contaminated data

This example compares the slope estimates from Gaussian, Huber, Laplace, and
Quantile fits on data with a few large outliers. The reference is the slope
used to generate the uncontaminated data.


In [1]:
import numpy as np
import mboostrs as mb

rng = np.random.default_rng(0)
n = 300
x = rng.normal(size=n)
true_slope = 2.0
y = 1.0 + true_slope * x + rng.normal(size=n) * 0.5
# Contaminate 10% of observations with large-magnitude outliers.
n_out = int(0.1 * n)
out_idx = rng.choice(n, n_out, replace=False)
y[out_idx] += rng.choice([-1, 1], n_out) * 50
data = {"y": y, "x": x}

errors = {}
for name, family in [("Gaussian", mb.Gaussian()), ("Huber", mb.Huber()),
                     ("Laplace", mb.Laplace()), ("Quantile (median)", mb.Quantile(tau=0.5))]:
    fit = mb.glmboost("y ~ bols(x)", data=data, family=family,
                       control=mb.boost_control(mstop=200, nu=0.1))
    coef = fit.coef()["x"]
    errors[name] = abs(coef - true_slope)
    print(f"{name:20s}: coef(x) = {coef:.4f}  (true = {true_slope}), "
          f"error = {errors[name]:.4f}")

# The claim under test: robust families recover the true slope more closely
# than Gaussian once the data are contaminated with large outliers.
for name in ("Huber", "Laplace", "Quantile (median)"):
    assert errors[name] < errors["Gaussian"], (
        f"{name} error {errors[name]:.4f} should be smaller than "
        f"Gaussian error {errors['Gaussian']:.4f} under contamination"
    )


Gaussian            : coef(x) = 2.4609  (true = 2.0), error = 0.4609
Huber               : coef(x) = 2.0285  (true = 2.0), error = 0.0285
Laplace             : coef(x) = 2.0168  (true = 2.0), error = 0.0168
Quantile (median)   : coef(x) = 2.0167  (true = 2.0), error = 0.0167


With 10% of observations perturbed by $\pm 50$ (100$\times$ the
clean-data noise scale), Gaussian's recovered slope is off by roughly
0.46 — the squared loss lets a handful of extreme points dominate the risk.
Huber, Laplace, and the median-quantile fit all recover the slope to within
about 0.03: their gradients saturate at $\pm\delta$ (Huber) or $\pm 1$
(Laplace, Quantile) regardless of how far an outlier sits from the fit.


## Summary

- The family defines the loss function $\rho(y, f)$, its negative gradient $u$,
  the empirical risk $\mathcal R$, and the inverse link. Changing the family
  retargets the same boosting algorithm to classification, robust regression,
  quantiles, counts, and survival.
- Offsets minimize the risk for the null model; `calibrate` updates per-step
  nuisance parameters (Huber's $\delta$, Student-t $\nu$).
- To add a new family, implement `impl Family for YourFamily` in Rust
  (`families.rs`) and register the Python wrapper in `families.py`.

## See also

- [](gamlss.ipynb) — distributional families (GAMLSS) with per-parameter gradients.
- [](survival.ipynb) — the survival families (CoxPH, Weibull, Loglog, Lognormal).
- [](../implementation/architecture) — the `Family` trait and the
  `with_concrete_family!` macro.
- [](../implementation/testing) — the finite-difference gradient verifier.